In [22]:
# ============================================================
# 1. INSTALLATION
# ============================================================

!pip install -q langchain-openai langchain-core requests


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import json
import requests

from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage
from typing import Annotated


# ============================================================
# 3. OPENAI API KEY
# ============================================================

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# ============================================================
# 4. MULTIPLY TOOL
# ============================================================

@tool
def multiply(a: int, b: int) -> int:
    """
    This function multiplies two numbers.
    """
    return a * b


# ============================================================
# 5. MULTIPLY TOOL BINDING
# ============================================================

llm_multiply = ChatOpenAI()

llm_multiply_with_tools = llm_multiply.bind_tools([multiply])


# ============================================================
# 6. MULTIPLY TOOL CALL
# ============================================================

messages = [
    HumanMessage(
        content="hi how are you? can you please multiply 4 and 5"
    )
]

ai_message = llm_multiply_with_tools.invoke(messages)

print("Multiply tool call:")
print(ai_message.tool_calls)

messages.append(ai_message)


# ============================================================
# 7. EXECUTE MULTIPLY TOOL
# ============================================================

for tool_call in ai_message.tool_calls:

    if tool_call["name"] == "multiply":

        tool_result = multiply.invoke(tool_call)

        print("\nMultiply tool result:")
        print(tool_result)

        messages.append(tool_result)


# ============================================================
# 8. FINAL MULTIPLY RESPONSE
# ============================================================

final_multiply_response = llm_multiply_with_tools.invoke(messages)

print("\nFinal multiply response:")
print(final_multiply_response.content)


# ============================================================
# 9. CURRENCY CONVERSION TOOLS
# ============================================================

@tool
def get_conversion_factor(
    base_currency: str,
    target_currency: str
) -> dict:
    """
    This function fetches the currency conversion factor
    between two currencies.
    """

    url = (
        f"https://v6.exchangerate-api.com/v6/"
        f"c12ec68d8ddc9f628a2e60d3/pair/"
        f"{base_currency}/{target_currency}"
    )

    response = requests.get(url)

    return response.json()


@tool
def convert(
    base_currency_value: int,
    conversion_rate: Annotated[float, InjectedToolArg]
) -> float:
    """
    Given a currency conversion rate this function calculates
    the converted currency value.
    """

    return base_currency_value * conversion_rate


# ============================================================
# 10. TEST CURRENCY API
# ============================================================

currency_response = get_conversion_factor.invoke(
    {
        "base_currency": "USD",
        "target_currency": "INR"
    }
)

print("\nCurrency API response:")
print(currency_response)


# ============================================================
# 11. TEST CONVERT TOOL
# ============================================================

direct_convert_result = convert.invoke(
    {
        "base_currency_value": 10,
        "conversion_rate": 85.16
    }
)

print("\nDirect convert test:")
print(direct_convert_result)


# ============================================================
# 12. BIND CURRENCY TOOLS
# ============================================================

llm1 = ChatOpenAI()

llm_with_tools = llm1.bind_tools(
    [
        get_conversion_factor,
        convert
    ]
)


# ============================================================
# 13. USER QUERY
# ============================================================

messages = [
    HumanMessage(
        content=(
            "what is the conversion rate between usd and inr "
            "and based on that can you convert 19 usd to inr"
        )
    )
]


# ============================================================
# 14. TOOL-CALLING LOOP
# ============================================================

conversion_rate = None

while True:

    # --------------------------------------------------------
    # Ask LLM what tool it wants to use
    # --------------------------------------------------------

    ai_message = llm_with_tools.invoke(messages)

    print("\nAI tool calls:")
    print(ai_message.tool_calls)

    # Add AI message to conversation
    messages.append(ai_message)


    # --------------------------------------------------------
    # If there are NO tool calls, this is the final response
    # --------------------------------------------------------

    if not ai_message.tool_calls:

        final_currency_response = ai_message

        break


    # --------------------------------------------------------
    # FIRST PASS:
    # Execute get_conversion_factor
    # --------------------------------------------------------

    for tool_call in ai_message.tool_calls:

        if tool_call["name"] == "get_conversion_factor":

            tool_message1 = get_conversion_factor.invoke(tool_call)

            print("\nConversion factor tool result:")
            print(tool_message1)

            # Extract conversion rate
            conversion_rate = json.loads(
                tool_message1.content
            )["conversion_rate"]

            print("\nConversion rate:")
            print(conversion_rate)

            # VERY IMPORTANT
            # Add ToolMessage to messages
            messages.append(tool_message1)


    # --------------------------------------------------------
    # SECOND PASS:
    # Execute convert IF the LLM already requested it
    # --------------------------------------------------------

    for tool_call in ai_message.tool_calls:

        if tool_call["name"] == "convert":

            # Inject conversion rate
            tool_call["args"]["conversion_rate"] = conversion_rate

            tool_message2 = convert.invoke(tool_call)

            print("\nConvert tool result:")
            print(tool_message2)

            # VERY IMPORTANT
            # Add ToolMessage to messages
            messages.append(tool_message2)


# ============================================================
# 15. FINAL CURRENCY RESPONSE
# ============================================================

print("\nFinal currency response:")
print(final_currency_response.content)

Multiply tool call:
[{'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'call_sWHQjcH78MzhtXxr8FhDBfGz', 'type': 'tool_call'}]

Multiply tool result:
content='20' name='multiply' tool_call_id='call_sWHQjcH78MzhtXxr8FhDBfGz'

Final multiply response:
The product of 4 and 5 is 20. Let me know if you need any more calculations!

Currency API response:
{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1790294401, 'time_last_update_utc': 'Fri, 25 Sep 2026 00:00:01 +0000', 'time_next_update_unix': 1790380801, 'time_next_update_utc': 'Sat, 26 Sep 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 96.0151}

Direct convert test:
851.5999999999999

AI tool calls:
[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_bO1szpv5nsJdTjn0drd9m4aS', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_